# Tahap 3 — Baseline Forecasting

**Research question:** Seberapa baik baseline persistence, historical-power Ridge, dan non-occupancy multivariate Ridge memprediksi power tepat 30 menit ke depan pada protokol temporal yang sama?

Notebook membaca hasil resmi Tahap 3 dan tidak melakukan retraining atau mengubah konfigurasi. Occupancy tidak digunakan sebagai feature pada tahap ini.

In [ ]:
# 2. Environment check
from pathlib import Path
import importlib.metadata
import os
import platform
import sys

EXPECTED_DATASET_SHA256 = 'ca7831a188a191edbf82a673fac90dbb875b5095986ed07699c02530f2a02a0e'
REPOSITORY_URL = 'https://github.com/rehanalfarizu/new_jurnal.git'
IS_COLAB = 'google.colab' in sys.modules

def find_repository_root():
    candidates = [Path.cwd(), Path.cwd() / 'new_jurnal', Path.cwd().parent]
    for candidate in candidates:
        if (candidate / 'configs' / 'experiment.yaml').is_file() and (candidate / 'src').is_dir():
            return candidate.resolve()
    return None

REPO_ROOT = find_repository_root()
print({'python': sys.version.split()[0], 'platform': platform.platform(), 'colab': IS_COLAB, 'repository_found': REPO_ROOT is not None})

In [ ]:
# 3. Google Colab setup
import subprocess

if IS_COLAB and REPO_ROOT is None:
    clone_target = Path('/content/new_jurnal')
    if not clone_target.exists():
        subprocess.run(['git', 'clone', REPOSITORY_URL, str(clone_target)], check=True)
    REPO_ROOT = clone_target.resolve()
elif REPO_ROOT is None:
    raise RuntimeError('Repository tidak ditemukan. Jalankan notebook dari root new_jurnal atau direktori notebooks/.')
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print('Repository aktif:', REPO_ROOT.name)

In [ ]:
# 4. Dependency setup
import importlib.util

if IS_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], check=True)
required_modules = {'numpy': 'numpy', 'pandas': 'pandas', 'scikit-learn': 'sklearn', 'matplotlib': 'matplotlib', 'PyYAML': 'yaml'}
missing = [name for name, module in required_modules.items() if importlib.util.find_spec(module) is None]
if missing:
    raise ModuleNotFoundError('Dependency kernel belum lengkap: ' + ', '.join(missing))
print('Dependency requirements.txt tersedia pada kernel aktif.')

In [ ]:
# 5. Dataset path + checksum
import hashlib
# Google Colab opsional:
# from google.colab import drive
# drive.mount('/content/drive')
# os.environ['SENSOR_DATA_PATH'] = '/content/drive/MyDrive/.../sensor_data.csv'
configured_path = os.environ.get('SENSOR_DATA_PATH', '').strip()
default_path = (REPO_ROOT / 'data/raw/sensor_data.csv').resolve()
if configured_path:
    DATASET_PATH = Path(configured_path).expanduser()
    if not DATASET_PATH.is_absolute():
        DATASET_PATH = (REPO_ROOT / DATASET_PATH).resolve()
    path_source = 'SENSOR_DATA_PATH'
elif default_path.is_file():
    DATASET_PATH, path_source = default_path, 'data/raw/sensor_data.csv'
elif not IS_COLAB:
    candidates_by_file = {}
    for pattern in ('*/Data/sensor_data.csv', '*/data/sensor_data.csv'):
        for path in REPO_ROOT.parent.glob(pattern):
            if path.is_file():
                info = path.stat()
                candidates_by_file[(info.st_dev, info.st_ino)] = path.resolve()
    candidates = sorted(candidates_by_file.values())
    if len(candidates) > 1:
        raise RuntimeError('Lebih dari satu dataset ditemukan; tetapkan SENSOR_DATA_PATH secara eksplisit.')
    DATASET_PATH = candidates[0] if candidates else default_path
    path_source = 'kandidat tunggal folder proyek saudara' if candidates else 'data/raw/sensor_data.csv'
else:
    DATASET_PATH, path_source = default_path, 'data/raw/sensor_data.csv'
if not DATASET_PATH.is_file():
    raise FileNotFoundError('sensor_data.csv tidak ditemukan. Tetapkan SENSOR_DATA_PATH atau mount Google Drive.')
def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()
DATASET_SHA256 = sha256_file(DATASET_PATH)
if DATASET_SHA256 != EXPECTED_DATASET_SHA256:
    raise ValueError(f'Checksum dataset tidak sesuai: {DATASET_SHA256}')
print('Sumber resolusi dataset:', path_source)
print('SHA-256 terverifikasi:', DATASET_SHA256)

In [ ]:
# 6. Load experiment configuration
import json
import pandas as pd
import yaml
from IPython.display import display
from src.features.time_series import BASELINE2_FEATURES, BASELINE3_FEATURES, FEATURE_DEFINITIONS, add_forecasting_features, add_power_target, resample_sensor_csv
from src.forecasting.pipeline import run_forecasting_foundation
from src.evaluation.metrics import regression_metrics
from src.evaluation.temporal import build_temporal_split, classify_modeling_samples

with Path('configs/experiment.yaml').open(encoding='utf-8') as handle:
    experiment_config = yaml.safe_load(handle)
with Path('configs/features.yaml').open(encoding='utf-8') as handle:
    feature_config = yaml.safe_load(handle)
with Path('results/metrics/forecast_run_manifest.json').open(encoding='utf-8') as handle:
    forecast_manifest = json.load(handle)
display(pd.DataFrame([
    {'setting': 'cadence_minutes', 'value': experiment_config['forecast']['cadence_minutes']},
    {'setting': 'horizon_minutes', 'value': experiment_config['forecast']['horizon_minutes']},
    {'setting': 'split_strategy', 'value': experiment_config['evaluation']['split_strategy']},
    {'setting': 'ridge_alphas', 'value': experiment_config['forecast']['ridge_alphas']},
    {'setting': 'occupancy_feature_used', 'value': forecast_manifest['occupancy_feature_used']},
]))
print('Pipeline source of truth:', run_forecasting_foundation.__module__ + '.' + run_forecasting_foundation.__name__)

In [ ]:
# 7. 1-minute resampling
modeling_summary = pd.read_csv('results/tables/modeling_dataset_summary.csv')
power_resampling = pd.read_csv('results/tables/power_resampling_comparison.csv')
display(modeling_summary[modeling_summary['metric'].isin(['raw_records', 'cadence_minutes', 'minute_bins_total', 'minute_bins_with_telemetry', 'minute_bins_complete'])])
display(power_resampling)
print('Resampling implementation:', resample_sensor_csv.__module__ + '.' + resample_sensor_csv.__name__)
print('Hasil resmi dibaca tanpa menjalankan ulang fitting model.')

In [ ]:
# 8. Missing minute-bin dan gap policy
display(modeling_summary[modeling_summary['metric'].isin(['minute_bins_missing', 'minute_bins_partial', 'samples_not_used'])])
display(pd.DataFrame([
    {'policy': 'gap_fill_policy', 'value': experiment_config['forecast']['gap_fill_policy']},
    {'policy': 'missing_policy', 'value': experiment_config['preprocessing']['missing_policy']},
    {'policy': 'boundary_policy', 'value': experiment_config['evaluation']['boundary_policy']},
]))

In [ ]:
# 9. Feature engineering
feature_definition = pd.read_csv('results/tables/feature_definition.csv')
display(feature_definition)
if feature_definition['uses_occupancy'].astype(bool).any():
    raise AssertionError('Notebook 03 tidak boleh menggunakan occupancy feature.')
if feature_config['model_feature_sets']['historical_power_ridge'] != BASELINE2_FEATURES:
    raise AssertionError('Feature historical-power tidak sesuai source code.')
if feature_config['model_feature_sets']['non_occupancy_multivariate_ridge'] != BASELINE3_FEATURES:
    raise AssertionError('Feature multivariate non-occupancy tidak sesuai source code.')

In [ ]:
# 10. Exact target power(t+30m)
test_predictions = pd.read_csv('results/metrics/test_predictions.csv')
source_time = pd.to_datetime(test_predictions['timestamp_utc'], utc=True)
target_time = pd.to_datetime(test_predictions['target_timestamp_utc'], utc=True)
target_delta_minutes = (target_time - source_time).dt.total_seconds() / 60
if not target_delta_minutes.eq(experiment_config['forecast']['horizon_minutes']).all():
    raise AssertionError('Ditemukan target timestamp yang tidak tepat t+30 menit.')
display(test_predictions[['timestamp_utc', 'target_timestamp_utc', 'actual_power_w']].head())
print('Seluruh target timestamp test tepat +30 menit:', bool(target_delta_minutes.eq(30).all()))

In [ ]:
# 11. Sample exclusion
sample_exclusion = pd.read_csv('results/tables/sample_exclusion_summary.csv')
display(sample_exclusion)
print('Sample dikeluarkan dari modeling eligibility; raw data dan minute grid tidak dihapus.')

In [ ]:
# 12. Chronological train/validation/test split
temporal_split = pd.read_csv('results/tables/temporal_split.csv')
display(temporal_split)
ordered_starts = pd.to_datetime(temporal_split['start_timestamp_utc'], utc=True)
if not ordered_starts.is_monotonic_increasing:
    raise AssertionError('Split tidak tersusun kronologis.')
print('Random split digunakan:', False)

In [ ]:
# 13. Persistence baseline
baseline_comparison = pd.read_csv('results/tables/baseline_model_comparison.csv')
persistence = baseline_comparison[baseline_comparison['model'] == 'persistence']
display(persistence)
if not (test_predictions['persistence_prediction_w'] == test_predictions['current_power_w']).all():
    raise AssertionError('Prediksi persistence tidak sama dengan current power.')

In [ ]:
# 14. Historical-power Ridge
historical_ridge = baseline_comparison[baseline_comparison['model'] == 'historical_power_ridge']
display(historical_ridge)
display(feature_definition[feature_definition['used_by_historical_power_ridge'].astype(bool)])

In [ ]:
# 15. Non-occupancy multivariate Ridge
multivariate_ridge = baseline_comparison[baseline_comparison['model'] == 'non_occupancy_multivariate_ridge']
display(multivariate_ridge)
display(feature_definition[feature_definition['used_by_non_occupancy_multivariate_ridge'].astype(bool)])
if forecast_manifest['occupancy_feature_used'] is not False:
    raise AssertionError('Manifest Tahap 3 harus menyatakan occupancy tidak digunakan.')

In [ ]:
# 16. MAE / RMSE / R² comparison
forecast_metrics = pd.read_csv('results/metrics/forecast_metrics.csv')
display(baseline_comparison[['model', 'validation_mae_w', 'validation_rmse_w', 'validation_r2', 'test_mae_w', 'test_rmse_w', 'test_r2', 'selected_from_validation_for_figures']])
display(forecast_metrics.pivot_table(index=['model', 'split'], columns='metric', values='value').reset_index())

In [ ]:
# 17. Actual-vs-predicted + forecast error
from IPython.display import Image
for figure_path in [
    'results/figures/forecast_actual_vs_predicted.png',
    'results/figures/forecast_error_distribution.png',
]:
    if not Path(figure_path).is_file():
        raise FileNotFoundError(figure_path)
    display(Image(filename=figure_path))

In [ ]:
# 18. Leakage and reproducibility summary
if not (feature_definition['source_offset_max'] <= 0).all():
    raise AssertionError('Ditemukan feature yang menggunakan future value.')
summary = {
    'dataset_sha256': DATASET_SHA256,
    'manifest_dataset_sha256': forecast_manifest['input_sha256'],
    'pipeline': run_forecasting_foundation.__module__ + '.' + run_forecasting_foundation.__name__,
    'feature_module': add_forecasting_features.__module__,
    'evaluation_modules': [regression_metrics.__module__, build_temporal_split.__module__],
    'maximum_feature_source_offset': int(feature_definition['source_offset_max'].max()),
    'occupancy_feature_used': forecast_manifest['occupancy_feature_used'],
    'model_retrained_in_notebook': False,
    'selected_for_figures_by_validation_mae': forecast_manifest['selected_for_figures_by_validation_mae'],
}
if forecast_manifest['input_sha256'] != DATASET_SHA256:
    raise AssertionError('Manifest forecasting tidak berasal dari dataset aktif.')
display(summary)